# W5D6 – Semantic Cache with Redis and Ollama

This notebook demonstrates semantic caching using 20 Q&A pairs, Ollama `nomic-embed-text`, Redis/Memurai, NumPy cosine similarity, and the required similarity threshold of 0.92.

Before running:
1. Start Memurai (Redis-compatible service) on `127.0.0.1:6379`.
2. Start Ollama.
3. Ensure `nomic-embed-text` is available.
4. Install the W5D6 requirements.
5. Run this notebook from the `W5D6_Cost_Optimisation` folder or ensure that folder is on Python's import path.


In [2]:
import json
import numpy as np
import requests
import redis

OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"
REDIS_HOST = "127.0.0.1"
REDIS_PORT = 6379
REDIS_PREFIX = "cynaris:w5d6:notebook:"
SIMILARITY_THRESHOLD = 0.92


In [3]:
import sys
print(sys.executable)

d:\Cynaris\.venv\Scripts\python.exe


In [4]:
QA_PAIRS = [
    ("What is machine learning?", "Machine learning enables systems to learn patterns from data."),
    ("Define supervised learning.", "Supervised learning trains on labelled examples."),
    ("What is unsupervised learning?", "Unsupervised learning discovers patterns in unlabelled data."),
    ("Explain reinforcement learning.", "Reinforcement learning learns through actions and rewards."),
    ("What is overfitting?", "Overfitting occurs when a model memorises training data and generalises poorly."),
    ("Define underfitting.", "Underfitting occurs when a model is too simple to learn data patterns."),
    ("What is a training dataset?", "A training dataset is used to fit a machine-learning model."),
    ("What is a test dataset?", "A test dataset evaluates a model on unseen examples."),
    ("Explain feature engineering.", "Feature engineering creates or transforms useful input variables."),
    ("What is classification?", "Classification predicts a category or class label."),
    ("What is regression?", "Regression predicts a continuous numerical value."),
    ("What is clustering?", "Clustering groups similar data points together."),
    ("What is an embedding?", "An embedding represents text as a numerical vector."),
    ("What is cosine similarity?", "Cosine similarity compares the direction of two vectors."),
    ("What is a vector database?", "A vector database stores and searches vector representations."),
    ("What is RAG?", "RAG retrieves relevant information and supplies it to a generative model."),
    ("What is a prompt?", "A prompt is the input or instruction supplied to a language model."),
    ("What is tokenisation?", "Tokenisation splits text into units that a model processes."),
    ("What is model evaluation?", "Model evaluation measures performance using suitable metrics."),
    ("What is data preprocessing?", "Data preprocessing cleans and transforms raw data before modelling.")
]
len(QA_PAIRS)


20

In [5]:
def get_embeddings(texts):
    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": EMBED_MODEL, "input": texts},
        timeout=300
    )
    response.raise_for_status()
    return response.json()["embeddings"]

redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    db=0,
    decode_responses=True
)
redis_client.ping()
print("Redis connection successful.")


Redis connection successful.


In [6]:
def cosine_similarity(vector_a, vector_b):
    a = np.asarray(vector_a, dtype=np.float32)
    b = np.asarray(vector_b, dtype=np.float32)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)


def store_qa_pair(question, answer, embedding):
    import hashlib
    key_hash = hashlib.sha256(question.encode("utf-8")).hexdigest()
    key = REDIS_PREFIX + key_hash
    payload = {
        "question": question,
        "answer": answer,
        "embedding": embedding
    }
    redis_client.set(key, json.dumps(payload), ex=86400)


def search_cache(query, query_embedding):
    best_item = None
    best_score = -1.0

    for key in redis_client.scan_iter(match=REDIS_PREFIX + "*"):
        raw_value = redis_client.get(key)
        if not raw_value:
            continue

        item = json.loads(raw_value)
        score = cosine_similarity(query_embedding, item["embedding"])

        if score > best_score:
            best_score = score
            best_item = item

    if best_item and best_score >= SIMILARITY_THRESHOLD:
        return {
            "cache_hit": True,
            "similarity": round(best_score, 4),
            "matched_question": best_item["question"],
            "answer": best_item["answer"]
        }

    return {
        "cache_hit": False,
        "best_similarity": round(best_score, 4)
    }


## Store the 20 Q&A pairs

The following cell clears only this notebook's own Redis key namespace. It does not delete other Redis data.


In [7]:
old_keys = list(redis_client.scan_iter(match=REDIS_PREFIX + "*"))
if old_keys:
    redis_client.delete(*old_keys)

questions = [question for question, _ in QA_PAIRS]
vectors = get_embeddings(questions)

for (question, answer), vector in zip(QA_PAIRS, vectors):
    store_qa_pair(question, answer, vector)

print(f"Stored {len(QA_PAIRS)} Q&A pairs in Redis.")


Stored 20 Q&A pairs in Redis.


## Test a semantic cache lookup

Start with an exact-match query to verify a reliable cache hit. Then try a paraphrase and inspect its similarity score.

In [8]:
test_query = "What is machine learning?"
test_vector = get_embeddings([test_query])[0]
result = search_cache(test_query, test_vector)
result


{'cache_hit': True,
 'similarity': 1.0,
 'matched_question': 'What is machine learning?',
 'answer': 'Machine learning enables systems to learn patterns from data.'}

In [9]:
paraphrase = "Can you explain machine learning?"
paraphrase_vector = get_embeddings([paraphrase])[0]
paraphrase_result = search_cache(paraphrase, paraphrase_vector)
paraphrase_result


{'cache_hit': False, 'best_similarity': 0.913}

## Interpretation

- A cache hit is returned only when cosine similarity is at least **0.92**.
- If the score is below 0.92, the application should retrieve fresh context and generate a new answer.
- The exact-match test should normally return a score very close to 1.0.
- A paraphrase may or may not reach 0.92; record the actual measured result rather than assuming it will.
- Embeddings are generated locally by Ollama. Redis stores the query, answer and embedding.
